# Extract DEX data from Dune

Pull swap-level prices, per-swap gas, mint/burn liquidity events, chain gas, and
hourly USD token prices for the pair, then save one folder of CSVs under `S.data_dir`.
All parameters come from `arblib.config.STUDY`; install deps with
`pip install -r requirements.txt`.

In [4]:
import os

from dotenv import load_dotenv

from arblib import config, data_io, dune_api
from arblib.config import STUDY as S
from arblib import kraken_api

load_dotenv()
headers = dune_api.make_headers(os.environ["DUNE_API_KEY"])
params = S.collection_params
params

{'start_ts': '2025-12-31 15:00:00',
 'end_ts': '2025-12-31 16:00:00',
 'token0': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'token1': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'chain': 'ethereum'}

## Swaps

One swap query per DEX, each left-joined with that DEX's per-swap gas on
block / pool / tx / event index, saved under `S.swaps_dir`.

In [ ]:
merge_keys = ["evt_block_number", "pool", "evt_tx_hash", "evt_index"]

df_pancake_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["pancake"], params, headers, "Pancake")
df_gas_pancake = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["pancake_gas_per_swap"], params, headers, "Gas pancake")
if not df_pancake_swap.empty and not df_gas_pancake.empty:
    df_pancake_swap = (df_pancake_swap.merge(df_gas_pancake, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

df_uniswap_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["uniswap"], params, headers, "Uniswap")
df_gas_uniswap = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["uniswap_gas_per_swap"], params, headers, "Gas uniswap")
if not df_uniswap_swap.empty and not df_gas_uniswap.empty:
    df_uniswap_swap = (df_uniswap_swap.merge(df_gas_uniswap, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

data_io.save_dataframes(
    {config.SWAP_FILES["df_uniswap"]: df_uniswap_swap, config.SWAP_FILES["df_pancake"]: df_pancake_swap},
    S.swaps_dir,
)

## Liquidity

Mint / burn events per pool, one query per DEX, saved under `S.liquidity_dir`.

In [ ]:
df_uniswap_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")
if not df_uniswap_liq.empty:
    df_uniswap_liq = df_uniswap_liq.sort_values("evt_block_number").reset_index(drop=True)
if not df_pancake_liq.empty:
    df_pancake_liq = df_pancake_liq.sort_values("evt_block_number").reset_index(drop=True)

data_io.save_dataframes(
    {config.LIQUIDITY_FILES["df_uniswap"]: df_uniswap_liq, config.LIQUIDITY_FILES["df_pancake"]: df_pancake_liq},
    S.liquidity_dir,
)

## Chain gas

Per-block base fee + utilization for the whole chain, saved under `S.gas_dir`.

In [5]:
df_gas_chain = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["chain_gas_price"], params, headers, "Gas price")
if not df_gas_chain.empty:
    df_gas_chain = df_gas_chain.sort_values("block_number").reset_index(drop=True)

data_io.save_dataframes({config.GAS_FILES["chain_gas_price"]: df_gas_chain}, S.gas_dir)

[Gas price] dropping params not used by query 7748900: ['token0', 'token1']
[Gas price] EXECUTE RESPONSE: {'execution_id': '01KY579PTRQCKMRSKCF5G0SPG8', 'state': 'QUERY_STATE_PENDING'}
[Gas price] STATUS: QUERY_STATE_EXECUTING
[Gas price] STATUS: QUERY_STATE_COMPLETED
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/gas/chain_gas_price.csv
Done.


## Kraken USD prices for X and Y

1-minute USD close prices for **X** (`token0`) and **Y** (`token1`) from the Kraken
**Trades** endpoint (historical, paginated by `since`; the OHLC endpoint only serves recent
candles), saved as `X_USD_prices.csv` / `Y_USD_prices.csv` under `S.prices_dir`. These are
the pipeline's USD price source — used to value trades in USD (`trade_sizes`) and to build
the volatility control (`transform`). The fetch starts at `S.cex_start_ts` (one averaging
window before `start_ts`) so the earliest swap has a full window of *past* CEX prices.

In [ ]:
x_prices = kraken_api.usd_prices_1min(config.KRAKEN_PAIRS[S.token0.symbol], S.cex_start_ts, S.end_ts)
y_prices = kraken_api.usd_prices_1min(config.KRAKEN_PAIRS[S.token1.symbol], S.cex_start_ts, S.end_ts)

data_io.save_dataframes({S.x_price_path.name: x_prices, S.y_price_path.name: y_prices}, S.prices_dir)